In [1]:
!pip install pymc arviz


In [2]:
import pandas as pd
import pymc as pm
import arviz as az


WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


In [3]:
df = pd.read_csv(r"Cleaned_Data/Data.csv")

In [4]:
df.head(5)

,Unnamed: 0,geography,geography code,Rural Urban,Cars: No cars or vans in household; measures: Value,Cars: 1 car or van in household; measures: Value,Cars: 2 cars or vans in household; measures: Value,Cars: 3 cars or vans in household; measures: Value,Cars: 4 or more cars or vans in household; measures: Value,Distance travelled to work: Less than 2km; measures: Value,...,Distance (Work from home),Distance < 10 km,10 km < Distance < 30 km,30 km < Distanc < 60 km,Unemployed,Economically active,Economically Inactive,cluster,P1,P2
0,0,Darlington,E06000005,Total,13052,20682,10450,1962,524,11433,...,4180,28175,9962,3529,4002,49441,23192,3,8.120296,-2.323253
1,1,County Durham,E06000047,Total,60926,96086,52740,10775,3276,36588,...,20652,106475,71001,14772,17013,230088,136695,0,8.916958,-3.204791
2,2,Hartlepool,E06000001,Total,14268,16573,7662,1535,396,8452,...,2473,22415,7391,2324,5194,38319,23291,0,9.162120,-4.377292
3,3,Middlesbrough,E06000002,Total,21488,22963,10207,1945,600,10393,...,3337,37606,5418,3223,7631,55951,36969,0,9.813269,-5.781488
4,4,Northumberland,E06000057,Total,30543,60875,36916,7671,2529,25944,...,17894,60003,43224,15421,10329,147939,74956,1,6.843403,-0.309240


In [5]:
df.columns.tolist()

['Unnamed: 0',
 'geography',
 'geography code',
 'Rural Urban',
 'Cars: No cars or vans in household; measures: Value',
 'Cars: 1 car or van in household; measures: Value',
 'Cars: 2 cars or vans in household; measures: Value',
 'Cars: 3 cars or vans in household; measures: Value',
 'Cars: 4 or more cars or vans in household; measures: Value',
 'Distance travelled to work: Less than 2km; measures: Value',
 'Distance travelled to work: 2km to less than 5km; measures: Value',
 'Distance travelled to work: 5km to less than 10km; measures: Value',
 'Distance travelled to work: 10km to less than 20km; measures: Value',
 'Distance travelled to work: 20km to less than 30km; measures: Value',
 'Distance travelled to work: 30km to less than 40km; measures: Value',
 'Distance travelled to work: 40km to less than 60km; measures: Value',
 'Distance travelled to work: 60km and over; measures: Value',
 'Distance travelled to work: Work mainly at or from home; measures: Value',
 'Distance travelled t

### Distance

In [6]:
df_dist = df[["Distance < 10 km", "10 km < Distance < 30 km",	"30 km < Distanc < 60 km"]]
df_dist = df_dist.div(df_dist.sum(axis=1), axis=0)


In [7]:
df_dist = df_dist.drop(["30 km < Distanc < 60 km"], axis=1)

### Age

In [8]:
df_age = df[['Young 0-19',
 'Working-age 20-64',
 'Retired-age 65-90']]
df_age = df_age.div(df_age.sum(axis=1), axis=0)


In [9]:
df_age = df_age.drop(["Young 0-19"], axis=1)

### Economic Status

In [10]:
df_eco = df[['Unemployed',
 'Economically active',
 'Economically Inactive']]
df_eco = df_eco.div(df_eco.sum(axis=1), axis=0)


In [11]:
df_eco = df_eco.drop(['Economically Inactive'], axis=1)

### Cars

In [12]:
df_car = df[['No Cars and Vans',
 '1 Car or Van',
 '(2, 3, 4) Car or Van']]
df_car = df_car.div(df_car.sum(axis=1), axis=0)


In [13]:
df_car = df_car.drop(['(2, 3, 4) Car or Van'], axis=1)

In [14]:
df_final = pd.concat([df_age, df_car, df_dist, df_eco], axis = 1)

In [15]:
df_final.head(5)

,Working-age 20-64,Retired-age 65-90,No Cars and Vans,1 Car or Van,Distance < 10 km,10 km < Distance < 30 km,Unemployed,Economically active
0,0.585806,0.174671,0.279666,0.443154,0.676211,0.239092,0.052222,0.645149
1,0.595290,0.179925,0.272230,0.429333,0.553842,0.369320,0.044328,0.599506
2,0.582529,0.169492,0.352871,0.409878,0.697635,0.230034,0.077750,0.573603
3,0.588569,0.149488,0.375645,0.401430,0.813155,0.117154,0.075892,0.556444
4,0.582970,0.200311,0.220473,0.439423,0.505723,0.364304,0.044288,0.634322


In [16]:
y = df_final["Economically active"]

In [17]:
X = df[["Working-age 20-64",	"Retired-age 65-90", 	"No Cars and Vans",	"1 Car or Van",	"Distance < 10 km",	"10 km < Distance < 30 km",	"Unemployed"]]

In [ ]:
with pm.Model() as employment_model:
    # Priors for the intercept and coefficients
    alpha = pm.Normal('intercept', mu=0, sigma=10)
    beta = pm.Normal('beta', mu=0, sigma=10, shape=X.shape[1])
    
    # Prior for the measurement error (must be positive)
    sigma = pm.Exponential('sigma', 1.0)
    
    # Expected value (Linear Regression formula)
    mu = alpha + pm.math.dot(X.values, beta)
    
    # Likelihood: The observed employment data
    likelihood = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y.values)
    
    # Inference: Draw 2,000 samples from the posterior
    trace = pm.sample(draws=1000, tune=1000, chains=2, random_seed=42)

# 3. View the Posterior thing
print(az.summary(trace, round_to=2))
az.plot_posterior(trace)


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [intercept, beta, sigma]


Output()